# SigTekX + AWS SageMaker: Anomaly Detection in Ionospheric Signals

This notebook demonstrates the ML pipeline integration of SigTekX with anomaly detection
for VLF/ULF ionospheric disturbance monitoring.

**Pipeline:**
1. Generate a synthetic VLF signal
2. Process through SigTekX CUDA STFT engine to get spectral magnitudes
3. Inject ionospheric disturbances (solar flares, geomagnetic storms)
4. Detect anomalies with IsolationForest (or SageMaker RCF)
5. Evaluate with precision/recall/F1

**Requirements:** SigTekX built with GPU support, scikit-learn

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import sigtekx

from scripts.aws.anomaly_detection import (
    generate_spectral_timeseries,
    inject_anomalies,
    run_local_detection,
    compute_detection_metrics,
    find_optimal_threshold,
)

print(f'SigTekX v{sigtekx.__version__}')

## 1. Generate Spectral Data via SigTekX CUDA Engine

We create a synthetic VLF signal with multiple carrier frequencies (typical of
ionosphere monitoring stations), then process it through SigTekX's STFT pipeline
on the GPU. This is the same engine used in our benchmark experiments.

In [ ]:
# Process 60s of signal through SigTekX (4096-pt FFT, 75% overlap)
times, spectral_data, n_bins = generate_spectral_timeseries(
    duration_sec=60.0,
    sample_rate_hz=48000.0,
    nfft=4096,
    overlap=0.75,
)

print(f'\nSpectral output shape: {spectral_data.shape}')
print(f'Time range: {times[0]:.2f}s to {times[-1]:.2f}s')

In [ ]:
# Visualize the STFT output
fig, ax = plt.subplots(figsize=(12, 4))
n_show = min(256, spectral_data.shape[1])
im = ax.imshow(
    spectral_data[:, :n_show].T, aspect='auto', origin='lower',
    extent=[times[0], times[-1], 0, n_show],
    cmap='viridis'
)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency Bin')
ax.set_title('SigTekX STFT Output (Normal VLF Signal)')
plt.colorbar(im, label='Magnitude')
plt.tight_layout()
plt.show()

## 2. Inject Ionospheric Disturbances

Three anomaly types simulating real ionospheric phenomena:
- **Power spike**: Broadband increase (Sudden Ionospheric Disturbance from solar flares)
- **Frequency shift**: Enhanced low-frequency power (geomagnetic storm onset)
- **Narrowband burst**: Single-frequency spike (VLF transmitter interference)

In [ ]:
# Split: 60% normal (train), 40% for testing (with injected anomalies)
split_idx = int(len(spectral_data) * 0.6)
train_data = spectral_data[:split_idx]
test_base = spectral_data[split_idx:]
test_times = times[split_idx:]

frame_rate = 1.0 / (times[1] - times[0])
test_data, anomaly_meta = inject_anomalies(
    test_base, test_times, frame_rate, n_anomalies=5
)

print(f'Training set: {train_data.shape[0]} normal frames')
print(f'Test set: {test_data.shape[0]} frames (with anomalies)\n')
for a in anomaly_meta:
    print(f'  {a["type"]:20s}  t={a["start_time"]:.2f}s-{a["end_time"]:.2f}s  '
          f'({a["end_frame"] - a["start_frame"]} frames)')

In [ ]:
# Compare normal vs anomalous spectral data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
n_show = min(256, test_base.shape[1])

ax1.imshow(test_base[:, :n_show].T, aspect='auto', origin='lower',
           extent=[test_times[0], test_times[-1], 0, n_show], cmap='viridis')
ax1.set_ylabel('Freq Bin')
ax1.set_title('Baseline (Normal SigTekX Output)')

ax2.imshow(test_data[:, :n_show].T, aspect='auto', origin='lower',
           extent=[test_times[0], test_times[-1], 0, n_show], cmap='viridis')
for a in anomaly_meta:
    ax2.axvspan(a['start_time'], a['end_time'], alpha=0.3, color='red')
ax2.set_ylabel('Freq Bin')
ax2.set_xlabel('Time (s)')
ax2.set_title('With Injected Disturbances (red regions)')

plt.tight_layout()
plt.show()

## 3. Anomaly Detection

Using **IsolationForest** as a local stand-in for SageMaker RCF.
Both are tree-based unsupervised anomaly detectors.

For cloud deployment, replace with `run_sagemaker_rcf()` for the AWS-native version.

In [ ]:
scores = run_local_detection(train_data, test_data)

print(f'Scored {len(scores)} frames')
print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]')

## 4. Results

In [ ]:
threshold = find_optimal_threshold(scores, anomaly_meta)
metrics = compute_detection_metrics(scores, anomaly_meta, threshold)

print('Detection Metrics:')
print(f'  Precision: {metrics["precision"]:.3f}')
print(f'  Recall:    {metrics["recall"]:.3f}')
print(f'  F1 Score:  {metrics["f1"]:.3f}')
print(f'  Threshold: {metrics["threshold"]:.4f} (auto-optimized)')
print(f'  TP={metrics["true_positives"]}  FP={metrics["false_positives"]}  FN={metrics["false_negatives"]}')

In [ ]:
# Full visualization
detected = scores > threshold
n_show = min(256, test_data.shape[1])

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Spectrogram
ax1 = axes[0]
im = ax1.imshow(test_data[:, :n_show].T, aspect='auto', origin='lower',
                extent=[test_times[0], test_times[-1], 0, n_show], cmap='viridis')
for a in anomaly_meta:
    ax1.axvspan(a['start_time'], a['end_time'], alpha=0.3, color='red')
ax1.set_ylabel('Frequency Bin')
ax1.set_title('SigTekX STFT Output with Injected Disturbances')
plt.colorbar(im, ax=ax1, label='Magnitude')

# Scores
ax2 = axes[1]
ax2.plot(test_times[:len(scores)], scores, color='darkred', linewidth=0.5)
ax2.axhline(y=threshold, color='orange', linestyle='--',
            label=f'Threshold ({threshold:.3f})')
for a in anomaly_meta:
    ax2.axvspan(a['start_time'], a['end_time'], alpha=0.2, color='red')
ax2.set_ylabel('Anomaly Score')
ax2.set_title('Anomaly Detection Scores')
ax2.legend()

# Detections
ax3 = axes[2]
ax3.fill_between(test_times[:len(scores)], detected.astype(float),
                 alpha=0.5, color='red', label='Detected')
for i, a in enumerate(anomaly_meta):
    ax3.axvspan(a['start_time'], a['end_time'], alpha=0.2, color='blue',
               label='Ground Truth' if i == 0 else None)
ax3.set_ylabel('Anomaly Detected')
ax3.set_xlabel('Time (seconds)')
ax3.set_title(f'Precision={metrics["precision"]:.2f}  Recall={metrics["recall"]:.2f}  F1={metrics["f1"]:.2f}')
ax3.set_ylim(-0.1, 1.5)
ax3.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 5. Summary

**What this demonstrates:**
- SigTekX CUDA STFT engine produces spectral data that feeds directly into ML pipelines
- Tree-based anomaly detectors effectively identify ionospheric disturbances
- The pipeline scales to cloud GPU instances via SageMaker Processing Jobs

**For SageMaker deployment:**
```python
from scripts.aws.anomaly_detection import run_sagemaker_rcf
scores = run_sagemaker_rcf(train_data, test_data, role_arn)
```